In [1]:
#Simulated Annealing
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MATCH = 2
MISMATCH = -1
GAP = -2

# Load sequences
def load_data(file):
    df = pd.read_csv(file)
    return df["sequence"].tolist()

# Pad sequences
def pad_sequences(seqs):
    max_len = max(len(s) for s in seqs)
    return [s.ljust(max_len, '-') for s in seqs]

# Fitness function
def fitness(alignment):
    alignment = np.array([list(seq) for seq in alignment])
    score = 0

    for col in alignment.T:
        unique, counts = np.unique(col, return_counts=True)
        for u, c in zip(unique, counts):
            if u == '-':
                score += GAP * c
            else:
                score += MATCH * c
    return score

# Generate neighbor solution
def mutate(alignment):
    new_align = alignment.copy()
    idx = random.randint(0, len(new_align)-1)
    seq = list(new_align[idx])

    pos = random.randint(0, len(seq)-1)
    if seq[pos] != '-':
        seq.insert(pos, '-')
    else:
        seq[pos] = random.choice("ACGT")

    new_align[idx] = "".join(seq)
    return pad_sequences(new_align)

# Simulated annealing
def simulated_annealing(sequences, temp=100, cooling=0.95, iterations=200):
    current = pad_sequences(sequences)
    best = current
    best_score = fitness(current)

    scores = []

    for i in range(iterations):
        neighbor = mutate(current)
        delta = fitness(neighbor) - fitness(current)

        if delta > 0 or random.random() < np.exp(delta / temp):
            current = neighbor

        if fitness(current) > best_score:
            best = current
            best_score = fitness(current)

        scores.append(best_score)
        temp *= cooling

    # Plot
    plt.plot(scores)
    plt.title("Simulated Annealing Progress")
    plt.xlabel("Iteration")
    plt.ylabel("Score")
    plt.show()

    return best, best_score

# Run
sequences = load_data("config.csv")
best, score = simulated_annealing(sequences)

print("Score:", score)

ModuleNotFoundError: No module named 'numpy'